# 2. 데이터 전처리

원본 시험 220행을 학습 가능한 로트 93행으로 만드는 아홉 단계.
구현은 `pipeline/preprocess.py` 의 `build()` 에 있다.

| 단계 | 내용 | 근거 |
|---|---|---|
| ① 제품 필터 | `4G100 CUP (CV5)` | 불량 집계에 이 제품만 있다 |
| ② 로트 정규화 | `6AB-1` → `6AB` | 없으면 매칭 0건 |
| ③ 벤토나이트 분리 | `28/1` → `28` | 한 칸에 두 값이 들어 있다 |
| ④ 규격 이탈 플래그 | 집계 **전** 행 단위 | 집계 후엔 개별 이탈이 묻힌다 |
| ⑤ 로트 단위 집계 | 측정값 평균·최대·최소 / 벤토 최빈값 | 2.5 |
| ⑥ 비율 변환 | `100 × 불량 / 생산` | 개수로 두면 생산량을 학습 |
| ⑦ 조인 | `lot_key` inner join | |
| ⑧ 변수 선별 | 31개 → 1개 | `3-모델학습` |
| ⑨ 타겟 이진화 | `불량률 > 4.61%` | |

In [ ]:
import sys, os, warnings
sys.path.insert(0, '/data/Project/boyoung/pipeline')
for v in ('OMP_NUM_THREADS', 'MKL_NUM_THREADS', 'OPENBLAS_NUM_THREADS'):
    os.environ.setdefault(v, '1')
warnings.filterwarnings('ignore')

import numpy as np, pandas as pd
import matplotlib.pyplot as plt, seaborn as sns
plt.rc('font', family='NanumGothic'); plt.rc('axes', unicode_minus=False)
sns.set_theme(style='whitegrid', font='NanumGothic', rc={'axes.unicode_minus': False})
pd.set_option('display.width', 180); pd.set_option('display.max_columns', 40)

import db, preprocess as P, classify as C

## 2.1 제품 필터

In [ ]:
f = db.table('foundry_test_result')
g = f[f['product_name'] == P.PRODUCT].copy()
print(f'전체 {len(f):,}행 · 제품 {f["product_name"].nunique()}종  →  {len(g)}행')
print('\n불량 집계에 있는 제품:', db.table('bad_statistics')['product_name'].unique())

## 2.2 로트번호 정규화

시험은 차수를 붙이고(`6AB-1`) 불량 집계는 안 붙인다(`6AB`). **이것이 없으면 매칭이 0건이다.**

In [ ]:
g['lot_key'] = P.lot_key(g['lot_no'])
print(g[['lot_no','lot_key']].drop_duplicates().head(8).to_string(index=False))
print()
import inspect; print(inspect.getsource(P.lot_key))

## 2.3 벤토나이트 분리

`bent_val` 은 `VARCHAR(30)` 이고 `28/1` 처럼 두 숫자가 빗금으로 묶여 있다.

In [ ]:
g = P._split_bent(g)
print(g[['lot_no','bent_val','bentonite']].head(6).to_string(index=False))
print()
bad = g[g['bentonite'].isna()][['lot_no','real_date','bent_val']]
print(f'형식이 안 맞는 행 {len(bad)} / {len(g)}')
print(bad.to_string(index=False) if len(bad) else '')
print('\n→ 28.1 은 빗금 대신 점. 수기 오류를 추측으로 메우지 않고 결측 처리한다.')

**뒤 숫자는 버린다.** 모래온도와 0.785 로 겹쳐 계절을 대신 담을 뿐이고, 넣으면 F1 이 0.531 에서 0.504 로 내려간다.

## 2.4 규격 이탈 플래그

규격이 전 품목 고정이라 '규격 대비 위치'는 원값의 선형변환일 뿐이다(상관 1.000). **경계에서 꺾이는 이탈 여부만** 원값과 다른 정보를 준다.

In [ ]:
g2 = P._add_spec_flags(g)
cols = ['lot_no','moist_val','moist_min','moist_max','moist_out']
print(g2[cols].head(6).to_string(index=False))
print()
for p in P.SPEC:
    print(f'  {P.ko(p+"_val_mean"):10} 이탈 {int(g2[f"{p}_out"].sum()):3}행 / {len(g2)}')

## 2.5 로트 단위 집계

In [ ]:
agg = P._aggregate(g2)
print(f'{len(g2)}행  →  {len(agg)}로트   컬럼 {agg.shape[1]}개')
print()
print([c for c in agg.columns][:14], '...')

### 벤토나이트만 최빈값인 이유

8종짜리 **이산 설정값**이라 평균을 내면 실재하지 않는 값이 생긴다.

In [ ]:
v = g2.groupby('lot_key')['bentonite'].agg(
    평균='mean', 최빈값=P._mode, 값_종류='nunique')
v.columns = ['평균', '최빈값', '값 종류']

print('값이 갈리는 로트')
print(v[v['값 종류'] > 1].round(2).to_string())
print(f'\n{len(v)}로트 중 {int((v["값 종류"]>1).sum())}개만 갈린다 — 98%가 한 값')
print()
print('평균을 쓰면 21 · 26.5 · 27.33 처럼 기록에 없는 값이 생긴다.')
print('로트 안에서 거의 변하지 않으므로 최대·최소도 만들지 않는다.')

In [ ]:
# 측정값은 다르다 — 로트 안에서 실제로 변한다
s = g2[g2['lot_key'] == '6DC'][['test_time','sand_temp','bentonite']]
print('로트 6DC'); print(s.to_string(index=False))

## 2.6 비율 변환과 조인

In [ ]:
data, report = P.build()
data = data.sort_values('target_date').reset_index(drop=True)
print(report)
print()
print(data[['lot_key','production_qty','bad_qty','bad_rate','bentonite']].head(6).to_string(index=False))
print(f'\n생산수량 {data["production_qty"].min():,} ~ {data["production_qty"].max():,}  '
      f'({data["production_qty"].max()/data["production_qty"].min():.1f}배)')

## 2.7 출력 구성

In [ ]:
feats = P.feature_columns(data)
print(f'{len(data)}행 × 변수 {len(feats)}개 + 타겟\n')
kind = {}
for c in feats:
    k = ('벤토나이트' if c == 'bentonite'
         else '규격 이탈' if c.endswith('_out_sum')
         else '측정 집계' if c.endswith(('_mean','_max','_min'))
         else '기타')
    kind.setdefault(k, []).append(c)
for k, v in kind.items():
    print(f'  {k:8} {len(v):2}개   {[P.ko(c) for c in v][:4]}{" ..." if len(v)>4 else ""}')

## 2.8 결측 처리 규칙

현재 데이터에는 결측이 없다. 아래는 **앞으로 들어올 데이터를 위한 규칙**이다.

In [ ]:
import inspect
print(inspect.getsource(C.impute))

| 대상 | 처리 | 근거 |
|---|---|---|
| 측정값 | 중앙값 + 결측 표시 | 이상치를 남겨두므로 평균보다 안전 |
| 규격 이탈 횟수 | **0** | 측정이 없으면 이탈을 *관측 못 한* 것이지 이탈이 아니다 |
| 로트 짝 없음 | 제외 (28로트) | 억지 매칭은 신호를 약화시킨다 |

**규격 이탈을 중앙값으로 채우면 안 된다.** `fswt_out_sum` 의 중앙값이 1 이라 없던 위반이 생긴다.

대체는 **분할 안에서** 계산한다. 전체 중앙값을 미리 구해 쓰면 검증 구간 정보가 학습에 새어든다.

## 2.9 타겟 이진화

In [ ]:
th = C.threshold(data['bad_rate'])
y = C.label(data['bad_rate'], th)
print(f'판정 기준  불량률 > {th:.2f}%  (로트별 불량률의 평균)')
print(f'양성 {int(y.sum())}  음성 {int((1-y).sum())}  기준선 정확도 {C.baseline_accuracy(y):.3f}')
print()
rows = []
for name, fn in C.THRESHOLDS.items():
    v = float(fn(data['bad_rate'])); yy = C.label(data['bad_rate'], v)
    rows.append({'기준': name, '값(%)': round(v,2), '양성': int(yy.sum()),
                 '양성 비율': f'{yy.mean():.0%}', '기준선 정확도': round(C.baseline_accuracy(yy),3)})
print(pd.DataFrame(rows).to_string(index=False))

**평균을 택했다.** 데이터에서 나온 값이라 근거를 설명할 수 있다. 현장에 불량률 관리 목표가 있다면 그 값으로 바꾸는 편이 실무적으로 맞다.